In [ ]:
 !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [7]:
 from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("/content/data/Python.txt")

/tmp/ipykernel_554/1554746289.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [8]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("/content/data/NIPS-2017-attention-is-all-you-need-Paper.pdf")

# document = pdf_loader.load()


## Ingestion Pipeline


In [9]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [10]:
# data -> documents
def load_all_pdfs():
    folder_path = "/content/data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"): # to choose only pdfs
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs +=1

    print("Total pdfs: ", num_docs)
    print("total pages: ", len(all_docs))
    return all_docs

In [11]:
all_pdf_documents = load_all_pdfs()

Total pdfs:  2
total pages:  32


In [ ]:
type(all_pdf_documents[1])

#chunks


In [13]:
# # chunks
# !pip install langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap = 50):

  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)

In [16]:
len(chunks)

321

In [17]:
#chunks

# Embeddings

In [19]:
from sentence_transformers import SentenceTransformer


In [23]:
# logic to create embeddings

In [21]:
class EmbeddingManager:
  def __init__(self, model_name = "all-MiniLM-L6-v2"):

    self.model_name = model_name
    print("loading model....", self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

  def generate_embedding(self, text):
    embeddings = self.model.encode(text, show_progress_bar = True)
    return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

###Vector Store


In [24]:
import chromadb
import uuid #for index

In [27]:
class VectorStoreManager:
  def __init__(self, persist_dictionary="data/vector_Store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_dictionary = persist_dictionary
    self.collection = None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_dictionary, exist_ok=True)

    # create a client
    self.client = chromadb.PersistentClient(path=self.persist_dictionary)

    #create a collection
    self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata = {"description": "vector store collection for pdf embeddings in RAG"})

    print("Initialized the vector store with collection: ", self.collection_name)
    print("docs in collection", self.collection.count())

  def add_documents(self, documents, embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("num of documents does not match num of embeddings")


    #store => ids, embeddings, document, metadata
    ids = []
    all_metadata= []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc_{str(uuid.uuid4())}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_length"] = 1
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding)

    self.collection.add(
        ids = ids,
        embeddings = embeddings_list,
        documents = documents_content,
        metadatas = all_metadata
    )


    print("total documents added in vector store=", len(documents_content))
    print("docs in collection", self.collection.count())

In [28]:
vector_store = VectorStoreManager()

Initialized the vector store with collection:  pdf_documents
docs in collection 0


In [30]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

total documents added in vector store= 321
docs in collection 321


#Retrieval pipeline

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

In [39]:
class RAGRetriever:
  def __init__(self, vector_store, embedding_manager):
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self, query, top_k = 5, score_threshold = 0.0):
    # generate embedding
    query_embedding = self.embedding_manager.generate_embedding(query)

    # semantic search in vector store
    results = self.vector_store.collection.query(
        query_embeddings = query_embedding,
        n_results = top_k
    )

    #cosine similarity

    retrieved_docs = []
    if results["documents"] and results["documents"][0]: # Changed 'document' to 'documents'
      ids = results["ids"][0]
      documents = results["documents"][0]
      metadatas = results["metadatas"][0]
      distances = results["distances"][0]

      for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
        similarity_score = 1-distance

        if similarity_score >= score_threshold:
          retrieved_docs.append(
              {
                  "id": doc_id,
                  "metadata": metadata,
                  "document": document,
                  "score": similarity_score,
                  "distance": distance,
                  "rank": i+1
              }
          )

      print(f"retrieved {len(retrieved_docs)} documents")

    else:
      print("no documents retrieved")

    return retrieved_docs

In [40]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [ ]:
rag_retriever.retrieve("What is RAG")